# UKGE & Energy-Based Baseline Experiments

**Goal**: Compare CAGP with KG-specific uncertainty methods.

Baselines:
1. **UKGE-style**: Triple-level confidence scores
2. **Energy-based OOD**: Use negative energy as uncertainty (Liu et al., NeurIPS 2020)

**Memory**: ~4GB GPU for FB15k-237

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, TensorDataset
from collections import defaultdict
import json
import os
import random

# Use local GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Using device: cuda
GPU: Tesla T4
Memory: 15.8 GB


In [2]:
CONFIG = {
    'epochs': 50,
    'embedding_dim': 100,  # Reduce if OOM
    'batch_size': 2048,
    'lr': 0.001,
    'seeds': [42, 123, 456],
}

In [3]:
import os

DATA_DIR = 'data/raw'
DATASET = 'fb15k-237'  # or 'wn18rr' (smaller)

# Download FB15K-237 if not present (for Colab)
if not os.path.exists(f'{DATA_DIR}/{DATASET}/train.txt'):
    print("Downloading FB15K-237 from Hugging Face...")
    os.makedirs(f'{DATA_DIR}/{DATASET}', exist_ok=True)
    !pip install -q datasets
    from datasets import load_dataset
    ds = load_dataset("KGraph/FB15k-237")

    for split, filename in [('train', 'train.txt'), ('validation', 'valid.txt'), ('test', 'test.txt')]:
        with open(f'{DATA_DIR}/{DATASET}/{filename}', 'w') as f:
            for row in ds[split]:
                # Dataset has 'text' column with space-separated h/r/t
                parts = row['text'].split()
                if len(parts) >= 3:
                    f.write(f"{parts[0]}\t{parts[1]}\t{parts[2]}\n")
    print("Downloaded FB15K-237")

def load_triples(path):
    triples = []
    with open(path) as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 3:
                triples.append((parts[0], parts[1], parts[2]))
    return triples

train = load_triples(f'{DATA_DIR}/{DATASET}/train.txt')
test = load_triples(f'{DATA_DIR}/{DATASET}/test.txt')

entities = set()
relations = set()
for h, r, t in train + test:
    entities.add(h)
    entities.add(t)
    relations.add(r)

ent2idx = {e: i for i, e in enumerate(entities)}
rel2idx = {r: i for i, r in enumerate(relations)}

print(f"Dataset: {DATASET}")
print(f"Entities: {len(entities)}, Relations: {len(relations)}")
print(f"Train: {len(train)}, Test: {len(test)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train.txt:   0%|          | 0.00/21.3M [00:00<?, ?B/s]

valid.txt: 0.00B [00:00, ?B/s]

test.txt: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/272115 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/17535 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/20466 [00:00<?, ? examples/s]

Downloaded FB15K-237
Dataset: fb15k-237
Entities: 14534, Relations: 237
Train: 272115, Test: 20466


## 1. UKGE-Style Model

UKGE learns confidence scores per triple. We adapt it for OOD detection:
- Low confidence = high uncertainty = more likely OOD

In [4]:
class UKGE(nn.Module):
    """
    Simplified UKGE: learns soft confidence for each triple.

    Score = sigmoid(DistMult(h, r, t)) interpreted as confidence.
    Uncertainty = 1 - confidence.
    """

    def __init__(self, num_entities, num_relations, dim):
        super().__init__()
        self.entity_emb = nn.Embedding(num_entities, dim)
        self.relation_emb = nn.Embedding(num_relations, dim)
        nn.init.xavier_uniform_(self.entity_emb.weight)
        nn.init.xavier_uniform_(self.relation_emb.weight)

    def forward(self, heads, relations, tails):
        h = self.entity_emb(heads)
        r = self.relation_emb(relations)
        t = self.entity_emb(tails)
        return (h * r * t).sum(dim=-1)

    def get_confidence(self, heads, relations, tails):
        """Get confidence score (higher = more confident)."""
        score = self.forward(heads, relations, tails)
        return torch.sigmoid(score)

    def get_uncertainty(self, heads, relations, tails):
        """Get uncertainty (higher = more uncertain = likely OOD)."""
        return 1 - self.get_confidence(heads, relations, tails)

## 2. Energy-Based OOD Detection

From Liu et al., NeurIPS 2020: use negative energy as OOD score.
Lower energy = ID, Higher energy = OOD.

In [5]:
class EnergyOOD(nn.Module):
    """
    Energy-based OOD detection for KGE.

    Energy = -logsumexp(score) over candidate tails.
    For simplicity, we use E = -score directly.
    """

    def __init__(self, num_entities, num_relations, dim):
        super().__init__()
        self.entity_emb = nn.Embedding(num_entities, dim)
        self.relation_emb = nn.Embedding(num_relations, dim)
        nn.init.xavier_uniform_(self.entity_emb.weight)
        nn.init.xavier_uniform_(self.relation_emb.weight)

    def forward(self, heads, relations, tails):
        h = self.entity_emb(heads)
        r = self.relation_emb(relations)
        t = self.entity_emb(tails)
        return (h * r * t).sum(dim=-1)

    def get_energy(self, heads, relations, tails, temperature=1.0):
        """
        Compute energy: E = -score/T
        Higher energy = more likely OOD.
        """
        score = self.forward(heads, relations, tails)
        return -score / temperature

    def get_uncertainty(self, heads, relations, tails):
        """Use energy as uncertainty."""
        return self.get_energy(heads, relations, tails)

## 3. Coverage + GP (CAGP) for Comparison

In [6]:
class CAGP(nn.Module):
    """Coverage-Augmented GP-KGE."""

    def __init__(self, num_entities, num_relations, dim):
        super().__init__()
        self.num_entities = num_entities
        self.dim = dim

        self.entity_mean = nn.Parameter(torch.randn(num_entities, dim) * 0.1)
        self.entity_logvar = nn.Parameter(torch.zeros(num_entities, dim) - 1.0)
        self.relation_emb = nn.Embedding(num_relations, dim)
        nn.init.xavier_uniform_(self.relation_emb.weight)

        self.register_buffer('coverage', torch.zeros(num_entities, num_relations))
        self.alpha_logit = nn.Parameter(torch.tensor(0.0))

    def forward(self, heads, relations, tails):
        if self.training:
            h = self._sample(heads)
            t = self._sample(tails)
        else:
            h = self.entity_mean[heads]
            t = self.entity_mean[tails]
        r = self.relation_emb(relations)
        return (h * r * t).sum(dim=-1)

    def _sample(self, indices):
        mean = self.entity_mean[indices]
        std = torch.exp(0.5 * self.entity_logvar[indices])
        return mean + std * torch.randn_like(std)

    def get_uncertainty(self, heads, relations, tails):
        h_var = torch.exp(self.entity_logvar[heads]).mean(dim=-1)
        t_var = torch.exp(self.entity_logvar[tails]).mean(dim=-1)
        gp_var = (h_var + t_var) / 2

        h_cov = self.coverage[heads, relations]
        t_cov = self.coverage[tails, relations]
        cov_unc = 2.0 - h_cov - t_cov

        gp_var_norm = gp_var / (gp_var.mean() + 1e-8) * cov_unc.mean()
        alpha = torch.sigmoid(self.alpha_logit)

        return alpha * gp_var_norm + (1 - alpha) * cov_unc

    def precompute_coverage(self, triples, ent2idx, rel2idx):
        for h, r, t in triples:
            self.coverage[ent2idx[h], rel2idx[r]] = 1.0
            self.coverage[ent2idx[t], rel2idx[r]] = 1.0

    def kl_loss(self):
        kl = -0.5 * torch.sum(
            1 + self.entity_logvar - self.entity_mean.pow(2) - self.entity_logvar.exp()
        )
        return kl / self.num_entities

## 4. Training

In [7]:
def train_model(model, triples, ent2idx, rel2idx, epochs, is_cagp=False):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])
    criterion = nn.BCEWithLogitsLoss()

    heads = torch.tensor([ent2idx[h] for h, r, t in triples])
    relations = torch.tensor([rel2idx[r] for h, r, t in triples])
    tails = torch.tensor([ent2idx[t] for h, r, t in triples])

    loader = DataLoader(
        TensorDataset(heads, relations, tails),
        batch_size=CONFIG['batch_size'], shuffle=True
    )

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch_h, batch_r, batch_t in loader:
            batch_h = batch_h.to(device)
            batch_r = batch_r.to(device)
            batch_t = batch_t.to(device)

            pos = model(batch_h, batch_r, batch_t)
            neg_t = torch.randint(0, len(ent2idx), batch_t.shape, device=device)
            neg = model(batch_h, batch_r, neg_t)

            loss = criterion(pos, torch.ones_like(pos)) + criterion(neg, torch.zeros_like(neg))

            if is_cagp:
                loss += 0.01 * model.kl_loss()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(loader):.4f}")

    return model


def evaluate_auroc(model, test_triples, ent2idx, rel2idx):
    model.eval()

    heads = torch.tensor([ent2idx.get(h, 0) for h, r, t in test_triples]).to(device)
    relations = torch.tensor([rel2idx.get(r, 0) for h, r, t in test_triples]).to(device)
    tails = torch.tensor([ent2idx.get(t, 0) for h, r, t in test_triples]).to(device)

    with torch.no_grad():
        id_unc = model.get_uncertainty(heads, relations, tails).cpu().numpy()

        # Random OOD
        ood_tails = torch.randint(0, len(ent2idx), tails.shape, device=device)
        ood_unc = model.get_uncertainty(heads, relations, ood_tails).cpu().numpy()

    labels = np.concatenate([np.zeros(len(id_unc)), np.ones(len(ood_unc))])
    scores = np.concatenate([id_unc, ood_unc])

    return roc_auc_score(labels, scores)

## 5. Main Experiment

In [8]:
results = {
    'UKGE': [],
    'Energy': [],
    'CAGP': [],
}

for seed in CONFIG['seeds']:
    print(f"\n{'='*60}")
    print(f"Seed {seed}")
    print('='*60)

    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    # 1. UKGE
    print("\n1. Training UKGE...")
    ukge = UKGE(len(ent2idx), len(rel2idx), CONFIG['embedding_dim'])
    ukge = train_model(ukge, train, ent2idx, rel2idx, CONFIG['epochs'])
    auroc = evaluate_auroc(ukge, test, ent2idx, rel2idx)
    results['UKGE'].append(auroc)
    print(f"   AUROC: {auroc:.4f}")

    # 2. Energy
    print("\n2. Training Energy-based...")
    energy = EnergyOOD(len(ent2idx), len(rel2idx), CONFIG['embedding_dim'])
    energy = train_model(energy, train, ent2idx, rel2idx, CONFIG['epochs'])
    auroc = evaluate_auroc(energy, test, ent2idx, rel2idx)
    results['Energy'].append(auroc)
    print(f"   AUROC: {auroc:.4f}")

    # 3. CAGP
    print("\n3. Training CAGP...")
    cagp = CAGP(len(ent2idx), len(rel2idx), CONFIG['embedding_dim'])
    cagp.precompute_coverage(train, ent2idx, rel2idx)
    cagp = train_model(cagp, train, ent2idx, rel2idx, CONFIG['epochs'], is_cagp=True)
    auroc = evaluate_auroc(cagp, test, ent2idx, rel2idx)
    results['CAGP'].append(auroc)
    print(f"   AUROC: {auroc:.4f}")
    print(f"   Learned alpha: {torch.sigmoid(cagp.alpha_logit).item():.4f}")

    # Clear GPU memory
    del ukge, energy, cagp
    torch.cuda.empty_cache()


Seed 42

1. Training UKGE...
  Epoch 10/50, Loss: 0.3789
  Epoch 20/50, Loss: 0.1597
  Epoch 30/50, Loss: 0.1094
  Epoch 40/50, Loss: 0.0858
  Epoch 50/50, Loss: 0.0713
   AUROC: 0.9917

2. Training Energy-based...
  Epoch 10/50, Loss: 0.3597
  Epoch 20/50, Loss: 0.1647
  Epoch 30/50, Loss: 0.1127
  Epoch 40/50, Loss: 0.0874
  Epoch 50/50, Loss: 0.0724
   AUROC: 0.9922

3. Training CAGP...
  Epoch 10/50, Loss: 1.5282
  Epoch 20/50, Loss: 1.4752
  Epoch 30/50, Loss: 1.4287
  Epoch 40/50, Loss: 1.3895
  Epoch 50/50, Loss: 1.3503
   AUROC: 0.9592
   Learned alpha: 0.5000

Seed 123

1. Training UKGE...
  Epoch 10/50, Loss: 0.3687
  Epoch 20/50, Loss: 0.1639
  Epoch 30/50, Loss: 0.1120
  Epoch 40/50, Loss: 0.0883
  Epoch 50/50, Loss: 0.0723
   AUROC: 0.9914

2. Training Energy-based...
  Epoch 10/50, Loss: 0.3542
  Epoch 20/50, Loss: 0.1627
  Epoch 30/50, Loss: 0.1117
  Epoch 40/50, Loss: 0.0877
  Epoch 50/50, Loss: 0.0718
   AUROC: 0.9921

3. Training CAGP...
  Epoch 10/50, Loss: 1.5283
 

In [9]:
print("\n" + "="*70)
print(f"FINAL RESULTS: {DATASET.upper()}")
print("="*70)

print(f"\n{'Method':<15} {'AUROC':<20}")
print("-"*35)
for method in results:
    mean = np.mean(results[method])
    std = np.std(results[method])
    print(f"{method:<15} {mean:.4f} ± {std:.4f}")

print("\n--- Analysis ---")
cagp_mean = np.mean(results['CAGP'])
best_baseline = max(np.mean(results['UKGE']), np.mean(results['Energy']))
best_name = 'UKGE' if np.mean(results['UKGE']) > np.mean(results['Energy']) else 'Energy'

if cagp_mean > best_baseline:
    gap = cagp_mean - best_baseline
    print(f"✓ CAGP beats {best_name} by {gap:.4f} ({gap/best_baseline*100:.1f}%)")
else:
    print(f"✗ {best_name} beats CAGP")


FINAL RESULTS: FB15K-237

Method          AUROC               
-----------------------------------
UKGE            0.9916 ± 0.0001
Energy          0.9922 ± 0.0001
CAGP            0.9598 ± 0.0004

--- Analysis ---
✗ Energy beats CAGP


In [10]:
# Save results
output = {
    'dataset': DATASET,
    'config': CONFIG,
    'results': {
        m: {'mean': float(np.mean(v)), 'std': float(np.std(v))}
        for m, v in results.items()
    }
}

with open(f'ukge_baseline_results_{DATASET}.json', 'w') as f:
    json.dump(output, f, indent=2)

print(f"\nResults saved to ukge_baseline_results_{DATASET}.json")


Results saved to ukge_baseline_results_fb15k-237.json
